# Section 09 lab — Handling null and missing values

Run the section setup once. Complete the exercises in order; each exercise uses a small DataFrame created in the notebook.


## Section setup — run once

Create one Spark session and import the expressions used throughout this section.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder.master('local[2]')
         .appName('PySpark-Null-Handling')
         .config('spark.sql.shuffle.partitions', '4')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
print('Spark:', spark.version, '| master:', spark.sparkContext.master)


## Exercise 1 — Find and count null values

Begin by measuring missing values without changing the data.


### Step 1 — Create customer rows

Use `None` to create SQL null values.


In [ ]:
customers = spark.createDataFrame([
    ('C01', 'Asha', 'Pune', 'asha@example.com'),
    ('C02', 'Ravi', None, 'ravi@example.com'),
    ('C03', 'Meera', 'Mumbai', None),
    ('C04', 'Kabir', None, None),
], ['customer_id', 'name', 'city', 'email'])
customers.show()


### Step 2 — Select rows with missing email addresses

`isNull` creates an explicit null condition.


In [ ]:
missing_email = customers.filter(F.col('email').isNull()).orderBy('customer_id')
missing_email.show()
assert missing_email.count() == 2


### Step 3 — Count nulls in every column

Convert each null test to 1 or 0 and add the results.


In [ ]:
null_counts = customers.select([F.sum(F.col(name).isNull().cast('int')).alias(name) for name in customers.columns])
null_counts.show()


### Paired literal SQL solution

Register the customer DataFrame and reproduce the filters and counts with SQL.


In [ ]:
customers.createOrReplaceTempView('s09_customers')
sql_missing = spark.sql("SELECT * FROM s09_customers WHERE email IS NULL ORDER BY customer_id")
sql_counts = spark.sql("""
SELECT
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id,
  SUM(CASE WHEN name IS NULL THEN 1 ELSE 0 END) AS name,
  SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS city,
  SUM(CASE WHEN email IS NULL THEN 1 ELSE 0 END) AS email
FROM s09_customers
""")
sql_missing.show()
sql_counts.show()
assert sql_missing.collect() == missing_email.collect()


## Exercise 2 — Drop rows using clear requirements

Compare mandatory-column and minimum-completeness rules.


### Step 1 — Create incomplete order rows

Different rows omit different fields.


In [ ]:
orders = spark.createDataFrame([
    ('O01', 'C01', 'P01', 2),
    ('O02', None, 'P02', 1),
    ('O03', 'C03', None, 4),
    ('O04', None, None, None),
], ['order_id', 'customer_id', 'product_id', 'quantity'])
orders.show()


### Step 2 — Require the customer ID

Only the listed subset decides whether a row is removed.


In [ ]:
required_customer = orders.dropna(subset=['customer_id']).orderBy('order_id')
required_customer.show()
assert required_customer.count() == 2


### Step 3 — Require at least three known fields

`thresh=3` keeps rows containing three or more non-null values.


In [ ]:
threshold_result = orders.dropna(thresh=3).orderBy("order_id")
threshold_result.show()
assert threshold_result.count() == 3


### Paired literal SQL solution

Use `IS NOT NULL` expressions to apply the same completeness threshold.


In [ ]:
orders.createOrReplaceTempView('s09_orders')
sql_result = spark.sql("""
SELECT * FROM s09_orders
WHERE (CASE WHEN order_id IS NOT NULL THEN 1 ELSE 0 END
     + CASE WHEN customer_id IS NOT NULL THEN 1 ELSE 0 END
     + CASE WHEN product_id IS NOT NULL THEN 1 ELSE 0 END
     + CASE WHEN quantity IS NOT NULL THEN 1 ELSE 0 END) >= 3
ORDER BY order_id
""")
sql_result.show()
assert sql_result.collect() == threshold_result.collect()


## Exercise 3 — Fill columns with type-appropriate defaults

Use different defaults for text, integer and decimal columns.


### Step 1 — Create optional profile values

The original DataFrame remains unchanged.


In [ ]:
profiles = spark.createDataFrame([
    ('C01', 'Pune', 5, 0.10),
    ('C02', None, 2, None),
    ('C03', 'Mumbai', None, 0.05),
    ('C04', None, None, None),
], ['customer_id', 'city', 'orders', 'discount'])
profiles.show()


### Step 2 — Define defaults by column

A dictionary keeps every replacement rule visible.


In [ ]:
defaults = {'city': 'UNKNOWN', 'orders': 0, 'discount': 0.0}
filled_profiles = profiles.fillna(defaults).orderBy("customer_id")


### Step 3 — Verify that target columns contain no nulls

Count only the columns covered by the rule.


In [ ]:
filled_profiles.show()
assert filled_profiles.filter('city IS NULL OR orders IS NULL OR discount IS NULL').count() == 0


### Paired literal SQL solution

Use `COALESCE` to return a default when a column is null.


In [ ]:
profiles.createOrReplaceTempView('s09_profiles')
sql_result = spark.sql("""
SELECT customer_id, COALESCE(city, 'UNKNOWN') AS city,
       COALESCE(orders, 0) AS orders, COALESCE(discount, 0.0) AS discount
FROM s09_profiles ORDER BY customer_id
""")
sql_result.show()
assert sql_result.collect() == filled_profiles.collect()


## Exercise 4 — Handle null and NaN numeric values

`NULL` and `NaN` represent different kinds of missing numeric information.


### Step 1 — Create measurements containing both forms

Python `float("nan")` creates a not-a-number value.


In [ ]:
measurements = spark.createDataFrame([
    ('M01', 9.5),
    ('M02', None),
    ('M03', float('nan')),
    ('M04', 7.0),
], ['measurement_id', 'score'])
measurements.show()


### Step 2 — Identify each condition

`isNull` finds null; `isnan` finds NaN.


In [ ]:
classified = measurements.select('*', F.col('score').isNull().alias('is_null'), F.isnan('score').alias('is_nan'))
classified.show()


### Step 3 — Replace both with zero

Use one explicit expression so the rule is easy to review.


In [ ]:
clean_scores = measurements.withColumn('score', F.when(F.col('score').isNull() | F.isnan('score'), F.lit(0.0)).otherwise(F.col('score'))).orderBy('measurement_id')
clean_scores.show()
assert clean_scores.filter(F.col('score').isNull() | F.isnan('score')).count() == 0


### Paired literal SQL solution

Use `IS NULL` and `isnan` in a SQL `CASE` expression.


In [ ]:
measurements.createOrReplaceTempView('s09_measurements')
sql_result = spark.sql("""
SELECT measurement_id,
       CASE WHEN score IS NULL OR isnan(score) THEN 0.0 ELSE score END AS score
FROM s09_measurements ORDER BY measurement_id
""")
sql_result.show()
assert sql_result.collect() == clean_scores.collect()


## Exercise 5 — Select the first available contact value

Choose a preferred value without dropping the row.


### Step 1 — Create multiple contact columns

Some customers have only one contact number.


In [ ]:
contacts = spark.createDataFrame([
    ('C01', '9000000001', '020-111111'),
    ('C02', None, '080-222222'),
    ('C03', None, None),
], ['customer_id', 'mobile_number', 'home_number'])
contacts.show()


### Step 2 — Apply the preference order

Mobile is preferred, then home, then a visible fallback.


In [ ]:
preferred = contacts.withColumn('contact_number', F.coalesce('mobile_number', 'home_number', F.lit('NOT PROVIDED')))


### Step 3 — Inspect the selected values

Keep the source columns so the choice can be checked.


In [ ]:
preferred.orderBy("customer_id").show(truncate=False)
assert preferred.filter(F.col('contact_number').isNull()).count() == 0


### Paired literal SQL solution

SQL `COALESCE` applies the same left-to-right preference.


In [ ]:
contacts.createOrReplaceTempView('s09_contacts')
sql_result = spark.sql("""
SELECT *, COALESCE(mobile_number, home_number, 'NOT PROVIDED') AS contact_number
FROM s09_contacts ORDER BY customer_id
""")
sql_result.show(truncate=False)
assert sql_result.collect() == preferred.orderBy("customer_id").collect()


## Exercise 6 — Aggregate known and missing values

Compare row counts with non-null value counts.


### Step 1 — Create product ratings

Some customers did not provide a rating.


In [ ]:
ratings = spark.createDataFrame([
    ('Audio', 5.0), ('Audio', None), ('Audio', 4.0),
    ('Electronics', None), ('Electronics', 3.0),
], ['category', 'rating'])
ratings.show()


### Step 2 — Calculate category measures

`count(*)` counts rows; `count(rating)` counts known ratings.


In [ ]:
rating_summary = (ratings.groupBy('category').agg(
    F.count('*').alias('all_rows'),
    F.count('rating').alias('known_ratings'),
    F.avg('rating').alias('average_known_rating'),
).orderBy('category'))


### Step 3 — Derive the missing count

Subtract known values from all rows.


In [ ]:
rating_summary = rating_summary.withColumn('missing_ratings', F.col('all_rows') - F.col('known_ratings'))
rating_summary.show()


### Paired literal SQL solution

Aggregate the same null-aware measures with SQL.


In [ ]:
ratings.createOrReplaceTempView('s09_ratings')
sql_result = spark.sql("""
SELECT category, COUNT(*) AS all_rows, COUNT(rating) AS known_ratings,
       AVG(rating) AS average_known_rating, COUNT(*) - COUNT(rating) AS missing_ratings
FROM s09_ratings GROUP BY category ORDER BY category
""")
sql_result.show()
assert sql_result.collect() == rating_summary.collect()


## Exercise 7 — Forward fill within each device

Carry the most recent known reading forward through ordered events.


### Step 1 — Create device readings

Each device has an independent history.


In [ ]:
readings = spark.createDataFrame([
    ('D01', '2026-11-01 09:00:00', 21.0),
    ('D01', '2026-11-01 10:00:00', None),
    ('D01', '2026-11-01 11:00:00', 22.5),
    ('D01', '2026-11-01 12:00:00', None),
    ('D02', '2026-11-01 09:00:00', None),
    ('D02', '2026-11-01 10:00:00', 18.0),
], ['device_id', 'event_time', 'temperature']).withColumn('event_time', F.to_timestamp('event_time'))
readings.orderBy("device_id", "event_time").show()


### Step 2 — Define the history window

The frame starts at the first device event and ends at the current row.


In [ ]:
history = Window.partitionBy('device_id').orderBy('event_time').rowsBetween(Window.unboundedPreceding, Window.currentRow)


### Step 3 — Carry the last known value forward

Leading nulls remain null because no earlier reading exists.


In [ ]:
forward_filled = readings.withColumn('temperature_filled', F.last('temperature', ignorenulls=True).over(history)).orderBy('device_id', 'event_time')
forward_filled.show()


### Paired literal SQL solution

Use a null-ignoring window value over the same ordered frame.


In [ ]:
readings.createOrReplaceTempView('s09_readings_forward')
sql_result = spark.sql("""
SELECT *, LAST_VALUE(temperature, true) OVER (
  PARTITION BY device_id ORDER BY event_time
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) AS temperature_filled
FROM s09_readings_forward ORDER BY device_id, event_time
""")
sql_result.show()
assert sql_result.collect() == forward_filled.collect()


## Exercise 8 — Backward fill from the next known value

Use a following-row frame when a later confirmed reading may fill an earlier gap.


### Step 1 — Create an independent set of ordered readings

This exercise can run without executing Exercise 7.


In [ ]:
future_readings = spark.createDataFrame([
    ('D01', '2026-11-01 09:00:00', None),
    ('D01', '2026-11-01 10:00:00', 21.5),
    ('D01', '2026-11-01 11:00:00', None),
    ('D01', '2026-11-01 12:00:00', 22.0),
    ('D02', '2026-11-01 09:00:00', 18.0),
    ('D02', '2026-11-01 10:00:00', None),
], ['device_id', 'event_time', 'temperature']).withColumn('event_time', F.to_timestamp('event_time'))
future_readings.orderBy("device_id", "event_time").show()


### Step 2 — Define the future window

The frame starts at the current row and ends at the last device event.


In [ ]:
future = Window.partitionBy('device_id').orderBy('event_time').rowsBetween(Window.currentRow, Window.unboundedFollowing)


### Step 3 — Select the next known value

A trailing null remains null when no later value exists.


In [ ]:
backward_filled = future_readings.withColumn('next_known_temperature', F.first('temperature', ignorenulls=True).over(future)).orderBy('device_id', 'event_time')
backward_filled.show()


### Paired literal SQL solution

Use `FIRST_VALUE` with nulls ignored over the following-row frame.


In [ ]:
future_readings.createOrReplaceTempView('s09_readings_backward')
sql_result = spark.sql("""
SELECT *, FIRST_VALUE(temperature, true) OVER (
  PARTITION BY device_id ORDER BY event_time
  ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
) AS next_known_temperature
FROM s09_readings_backward ORDER BY device_id, event_time
""")
sql_result.show()
assert sql_result.collect() == backward_filled.collect()
